In [1]:
from __future__ import annotations
 
import json
import logging
import time
import sys
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional

ROOT: Path = Path.cwd().parent
sys.path.append(str(ROOT))

from src.core.api.api_getter import YouTubeClient
from src.core.models import LimitsConfig, Video, Channel
from src.core.models import RateLimitConfig


client = YouTubeClient(
        rate_limit=RateLimitConfig(requests_per_second=3.0, burst=2),
        limits=LimitsConfig(max_videos=2000, max_comments_per_video=50),
)


AttributeError: 'Field' object has no attribute 'mkdir'

In [ ]:
import csv
from pathlib import Path
from typing import Dict, Optional, Any, List
import datetime
import os
import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)
logger: logging.Logger = logging.getLogger(__name__)

canais_tech_br_expandida: list[str] = [
    # Notícias e tecnologia geral
    "@canaltech",
    "@tecmundo",
    "@olhardigital",
    "@adrenaline",
    "@gesieltaveira",

    # Smartphones e eletrônicos
    "@leonardomuller",
    "@rafaeltech",
    "@bgeeks",
    "@mobizoo",
    "@tecnoblog",

    # Hardware e PC
    "@adrenaline",
    "@peperaiohardware",
    "@mwinformatica",
    "@pichauinfo",
    "@pcfacts",

    # Programação e desenvolvimento
    "@FilipeDeschamps",
    "@cursoemvideo",
    "@codigofontetv",
    "@Rocketseat",
    "@lucasmontano",     # Padronizado para minúsculas
    "@kipperdev",        # Substitui @FernandaKipper
    "@DevDojoBrasil",
    "@baltaio",
    "@DiasdeDev",

    # Linux e Open Source
    "@Diolinux",
    "@linuxtips",
    "@BosonTreinamentos", # Acento removido (era @BósonTreinamentos)

    # Ciência da Computação / Conteúdo técnico
    "@flutterando",
    "@manodeyvin",

    # Infraestrutura, Cloud e DevOps
    "@linuxtips",
    "@fullcycle",

    # IA e tendências
    "@lucasmontano",
    "@manodeyvin",
    "@pasquadev"
]

def criar_pasta_execucao(base_dir: str = "saida_youtube") -> Path:
    """
    Cria uma pasta com timestamp no formato:
    saida_youtube/2026-06-16_14-32/
    """
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    pasta = Path(base_dir) / stamp
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta


def abrir_csv_writer(
    caminho: Path,
    fieldnames: List[str],
) -> tuple[Any, csv.DictWriter]:
    """
    Abre arquivo CSV em append e escreve cabeçalho se ainda não existir.
    """
    existe = caminho.exists()
    f = open(caminho, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    if not existe:
        writer.writeheader()
    return f, writer

def carregar_cache_canais(caminho_csv: Path) -> Dict[str, str]:
    """Lê canais.csv e retorna {input_canal: channel_id}."""
    cache = {}
    with open(caminho_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            nome = row.get('input_canal')
            cid = row.get('channel_id')
            if nome and cid:
                cache[nome] = cid
    return cache


def deduplicar_preservando_ordem(items: List[str]) -> List[str]:
    return list(dict.fromkeys(items))



def fetch_top_comments_by_video_id(
    self,
    video_id: str,
    max_results: int = 30,
    order: str = "relevance",
) -> List[Dict[str, Any]]:
    """
    Retorna os comentários top-level mais relevantes de um vídeo.
    Economia de quota:
    - não busca replies
    - limita a 30
    - usa order=relevance
    """
    if not video_id:
        return []

    if not (1 <= max_results <= 100):
        raise ValueError("max_results deve estar entre 1 e 100.")

    comentarios: List[Dict[str, Any]] = []
    page_token: Optional[str] = None

    while len(comentarios) < max_results:
        per_page = min(100, max_results - len(comentarios))
        params: Dict[str, Any] = {
            "key": self._api_key,
            "part": "snippet",
            "videoId": video_id,
            "maxResults": per_page,
            "order": order,
            "textFormat": "plainText",
            "fields": (
                "nextPageToken,items(id,snippet("
                "topLevelComment(id,snippet(authorDisplayName,authorChannelId,value,likeCount,publishedAt)),"
                "totalReplyCount"
                "))"
            ),
        }
        if page_token:
            params["pageToken"] = page_token

        data: Dict[str, Any] = self._get(self._comment_threads_url, params)

        for item in data.get("items", []):
            top = item.get("snippet", {}).get("topLevelComment", {})
            sn = top.get("snippet", {}) if isinstance(top, dict) else {}

            comentarios.append({
                "video_id": video_id,
                "comment_thread_id": item.get("id"),
                "comment_id": top.get("id"),
                "author_display_name": sn.get("authorDisplayName"),
                "author_channel_id": (sn.get("authorChannelId") or {}).get("value"),
                "text": sn.get("value"),
                "like_count": sn.get("likeCount"),
                "published_at": sn.get("publishedAt"),
                "reply_count": item.get("snippet", {}).get("totalReplyCount", 0),
            })

            if len(comentarios) >= max_results:
                break

        page_token = data.get("nextPageToken")
        if not page_token:
            break

    return comentarios


In [ ]:
def executar_pipeline(
    client,
    canais: List[str],
    videos_por_canal: int = 30,
    comentarios_por_video: int = 10,
    coletar_comentarios: bool = True,
    max_videos_comentados_por_canal: Optional[int] = None,
    base_dir_saida: str = "saida_youtube",
    caminho_cache_canais: Optional[Path] = None,   # NOVO
    atualizar_info_canais: bool = False,           # NOVO
) -> Path:
    pasta_execucao = criar_pasta_execucao(base_dir_saida)

    arquivo_canais = pasta_execucao / "canais.csv"
    arquivo_videos = pasta_execucao / "videos.csv"
    arquivo_comentarios = pasta_execucao / "comentarios.csv"
    arquivo_erros = pasta_execucao / "erros.csv"

    canais = deduplicar_preservando_ordem(canais)

    cache_ids = {}
    if caminho_cache_canais and caminho_cache_canais.exists():
        cache_ids = carregar_cache_canais(caminho_cache_canais)
        logger.info("Cache de canais carregado: %d handles.", len(cache_ids))
    
    f_canais = f_videos = f_comentarios = f_erros = None
    writer_canais = writer_videos = writer_comentarios = writer_erros = None
    try:

        timestamp_execucao = datetime.now().isoformat(timespec="seconds")

        f_erros, writer_erros = abrir_csv_writer(
            arquivo_erros,
            ["entidade", "chave", "erro", "timestamp_execucao"]
        )

        f_canais, writer_canais = abrir_csv_writer(
            arquivo_canais,
            [
                "input_canal",
                "channel_id",
                "title",
                "custom_url",
                "description",
                "published_at",
                "country",
                "subscriber_count",
                "view_count",
                "video_count",
                "hidden_subscriber_count",
                "timestamp_execucao",
            ],
        )

        f_videos, writer_videos = abrir_csv_writer(
            arquivo_videos,
            [
                "input_canal",
                "channel_id",
                "video_id",
                "title",
                "description",
                "published_at",
                "channel_title",
                "view_count",
                "like_count",
                "comment_count",
                "duration",
                "url",
                "raw_json",
                "timestamp_execucao",
            ],
        )

        f_comentarios, writer_comentarios = abrir_csv_writer(
            arquivo_comentarios,
            [
                "input_canal",
                "channel_id",
                "video_id",
                "comment_thread_id",
                "comment_id",
                "author_display_name",
                "author_channel_id",
                "text",
                "like_count",
                "published_at",
                "reply_count",
                "timestamp_execucao",
            ],
        )
        timestamp_execucao = datetime.now().isoformat(timespec="seconds")

        for canal in canais:
            channel_id = cache_ids.get(canal)
            if not channel_id or atualizar_info_canais:
                if channel_id:
                    canal_obj_list = client.fetch_channels_by_ids([channel_id])
                else:
                    canais_encontrados = client.search_channels(
                        query=canal.replace("@", ""),
                        max_results=10, pages=1, max_items=1
                    )
                    canal_obj_list = canais_encontrados

                if not canal_obj_list:
                    raise ValueError(f"Nenhum canal encontrado para {canal}")

                canal_obj = canal_obj_list[0]
                canal_dict = canal_obj.to_dict() if hasattr(canal_obj, "to_dict") else canal_obj.__dict__
                channel_id = canal_dict.get("channel_id") or canal_dict.get("id")

                writer_canais.writerow({
                    "input_canal": canal,
                    "channel_id": canal_dict.get("channel_id") or canal_dict.get("id"),
                    "title": canal_dict.get("title") or canal_dict.get("name"),
                    "custom_url": canal_dict.get("custom_url") or canal_dict.get("customUrl"),
                    "description": canal_dict.get("description"),
                    "published_at": canal_dict.get("published_at") or canal_dict.get("publishedAt"),
                    "country": canal_dict.get("country"),
                    "subscriber_count": canal_dict.get("subscriber_count") or canal_dict.get("subscriberCount"),
                    "view_count": canal_dict.get("view_count") or canal_dict.get("viewCount"),
                    "video_count": canal_dict.get("video_count") or canal_dict.get("videoCount"),
                    "hidden_subscriber_count": canal_dict.get("hidden_subscriber_count") or canal_dict.get("hiddenSubscriberCount"),
                    "timestamp_execucao": timestamp_execucao,
                })
                if hasattr(client, "fetch_videos_by_channel_id"):
                    videos = client.fetch_videos_by_channel_id(
                        channel_id, max_results=videos_por_canal
                    )
                else:
                    videos_retornados = client.fetch_videos_by_channel_names_(
                        channel_names=[canal],
                        videos_per_channel=videos_por_canal,
                    )
                    
                    videos = videos_retornados.get(canal, [])
                if not videos:
                    print(f"Nenhum vídeo encontrado para {canal}")
                    continue
                if max_videos_comentados_por_canal is not None:
                    videos_para_comentarios = videos[:max_videos_comentados_por_canal]
                else:
                    videos_para_comentarios = videos

                for video in videos:
                    vdict = video.to_dict() if hasattr(video, "to_dict") else video.__dict__

                    video_id = vdict.get("video_id") or vdict.get("id")
                    channel_id = vdict.get("channel_id") or vdict.get("channelId") or writer_canais.fieldnames and canal_dict.get("channel_id")
                    title = vdict.get("title")
                    description = vdict.get("description")
                    published_at = vdict.get("published_at") or vdict.get("publishedAt")
                    channel_title = vdict.get("channel_title") or vdict.get("channelTitle")
                    view_count = vdict.get("view_count") or vdict.get("viewCount")
                    like_count = vdict.get("like_count") or vdict.get("likeCount")
                    comment_count = vdict.get("comment_count") or vdict.get("commentCount")
                    duration = vdict.get("duration")
                    url = vdict.get("url") or f"https://www.youtube.com/watch?v={video_id}"

                    writer_videos.writerow({
                        "input_canal": canal,
                        "channel_id": channel_id,
                        "video_id": video_id,
                        "title": title,
                        "description": description,
                        "published_at": published_at,
                        "channel_title": channel_title,
                        "view_count": view_count,
                        "like_count": like_count,
                        "comment_count": comment_count,
                        "duration": duration,
                        "url": url,
                        "raw_json": str(vdict),
                        "timestamp_execucao": timestamp_execucao,
                    })

                    if coletar_comentarios and video in videos_para_comentarios and video_id:
                        try:
                            comentarios = client.fetch_top_comments_by_video_id(
                                video_id=video_id,
                                max_results=comentarios_por_video,
                                order="relevance",
                            )

                            for c in comentarios:
                                writer_comentarios.writerow({
                                    "input_canal": canal,
                                    "channel_id": channel_id,
                                    "video_id": video_id,
                                    "comment_thread_id": c.get("comment_thread_id"),
                                    "comment_id": c.get("comment_id"),
                                    "author_display_name": c.get("author_display_name"),
                                    "author_channel_id": c.get("author_channel_id"),
                                    "text": c.get("text"),
                                    "like_count": c.get("like_count"),
                                    "published_at": c.get("published_at"),
                                    "reply_count": c.get("reply_count"),
                                    "timestamp_execucao": timestamp_execucao,
                                })

                        except Exception as e:
                            writer_erros.writerow({
                                "entidade": "comentarios",
                                "chave": f"{canal}:{video_id}",
                                "erro": str(e),
                                "timestamp_execucao": timestamp_execucao,
                            })

                f_canais.flush()
                f_videos.flush()
                f_comentarios.flush()

                print(f"{canal}: canais, vídeos e comentários salvos")

    except Exception as e:
        writer_erros.writerow({
            "entidade": "canal",
            "chave": canal,
            "erro": str(e),
            "timestamp_execucao": timestamp_execucao,
        })
        f_erros.flush()
        print(f"ERRO em {canal}: {e}")

    finally:
        for f in [f_canais, f_videos, f_comentarios, f_erros]:
            try:
                if f:
                    f.close()
            except Exception:
                pass

    print(f"Concluído. Saída em: {pasta_execucao}")
    return pasta_execucao


In [ ]:
pasta = executar_pipeline(
    client=client,
    canais=canais_tech_br_expandida,
    videos_por_canal=50,
    comentarios_por_video=30,
    coletar_comentarios=True,
    max_videos_comentados_por_canal=None,  # coloque 5 ou 10 para economizar mais quota
    base_dir_saida="saida_youtube",
)